# Modell zur Ausgabenkategorisierung

Das Modell zur Ausgabenkategorisierung verfolgt einen hybriden Ansatz, der regelbasierte Klassifikation, einen persistenten Speicher sowie ein lokal ausgeführtes Large Language Model (LLM) kombiniert. Ziel ist es, Banktransaktionen möglichst präzise zu kategorisieren und gleichzeitig die Anzahl der LLM-Abfragen zu minimieren.

Der Klassifikationsprozess besteht aus den folgenden Schritten:

* **Vorverarbeitung der Transaktion:** Der Zahlungsempfänger (Payee) wird normalisiert, indem Groß- und Kleinschreibung vereinheitlicht sowie Umlaute, Zahlen, Satzzeichen, URLs und Ortsnamen entfernt werden. Dadurch können unterschiedliche Schreibweisen desselben Händlers zuverlässig erkannt werden.
* **Speicherabfrage:** Der normalisierte Händlername wird mit einer lokalen SQLite-Datenbank verglichen, die bereits kategorisierte Händler enthält. Bei einer Übereinstimmung wird die gespeicherte Kategorie direkt übernommen.
* **Regelbasierte Klassifikation:** Ist kein Eintrag vorhanden, werden vordefinierte Schlüsselwortregeln für häufig vorkommende Händler (z. B. Supermärkte, Streaming-Dienste oder Verkehrsunternehmen) angewendet.
* **LLM als Fallback:** Kann die Transaktion weder durch den Speicher noch durch die Regeln kategorisiert werden, wird ein lokal über Ollama ausgeführtes LLM verwendet (qwen2.5:3b). Dieses ordnet die Transaktion einer der vorgegebenen Ausgabenkategorien zu.
* **Aktualisierung des Speichers:** Neu klassifizierte Händler werden in der SQLite-Datenbank gespeichert, sodass zukünftige Transaktionen desselben Händlers direkt aus dem Speicher kategorisiert werden können.

Zusätzlich unterstützt das System manuelle Korrekturen. Wird eine Kategorie durch den Benutzer angepasst, wird der entsprechende Eintrag in der Datenbank aktualisiert. Dadurch werden zukünftige Transaktionen desselben Händlers automatisch mit der korrigierten Kategorie versehen. Auf diese Weise passt sich das System kontinuierlich an das individuelle Ausgabeverhalten des Nutzers an, während sämtliche Finanzdaten lokal auf dem Gerät verbleiben.


In [1]:
# ollama pull qwen2.5:3b

import sqlite3
import json
import re
from pathlib import Path

import pandas as pd
import ollama
import geonamescache
import unicodedata

# import DKB parser temporarily to get real data
import sys
sys.path.append("../../")
from backend.parsers import DKB

gc = geonamescache.GeonamesCache()

# List of city names for filtering out from payee strings
city_names = {
    "".join(
        c for c in unicodedata.normalize("NFKD", city["name"].lower())
        if not unicodedata.combining(c)
    )
    for city in gc.get_cities().values()
}

## Datenbank
Die Datenbank kann im voraus initialisiert und mit Kategorien und Keywords bestückt werde, welche dann von dem LLM verwendet werden um die Ausgaben zu Kategorisieren. Dafür kann die Datei __init_categories.py__ entsprechend angepasst werden. Um die Datenbank zu initialisieren muss folgender Befehl im Terminal ausgeführt werden:  
`python init_categories.py `

The Class itself is stored in the file Expanse_Class.py

In [2]:
from Expanse_Class import ExpenseDatabase

## Weitere Funktionsdefinitionen

In [3]:
# ----------------------------------------------------------
# Text cleaning
# ----------------------------------------------------------

def clean_payee(text: str) -> str:
    """
    Normalizes a payee string by:
      - converting to lowercase
      - removing accents/umlauts
      - removing URLs
      - removing standalone numbers
      - removing special characters and punctuation
      - removing city names
      - collapsing multiple spaces
    """

    if pd.isna(text):
        return ""

    # Lowercase and strip whitespace
    text = str(text).lower().strip()

    # Remove accents (ä -> a, é -> e, ...)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    # Remove URLs/domains
    text = re.sub(
        r"\.com\b|\.co\.uk\b|www\.",
        " ",
        text
    )

    # Remove standalone numbers
    text = re.sub(
        r"\b\d+\b",
        " ",
        text
    )

    # Remove special characters
    text = re.sub(
        r"[*#]",
        " ",
        text
    )

    # Remove punctuation
    text = re.sub(
        r"[^\w\s]",
        " ",
        text
    )

    # Collapse whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()


    # Remove city names
    words = text.split()

    text = " ".join(
        word for word in words
        if word not in city_names
    )

    return text



# ----------------------------------------------------------
# LLM fallback
# ----------------------------------------------------------

def classify_with_llm(description, db):

    prompt = f"""
You are a transaction classification engine for personal finance.

Your task is to classify ONE German bank transaction into exactly ONE category.

The transaction description comes from a German bank export and may contain irrelevant banking information.

Classification process:

1. Ignore:
- payment methods (Kartenzahlung, Lastschrift, Überweisung, SEPA)
- payment providers (PAYPAL, VR PAYMENT, SUMUP, STRIPE)
- transaction IDs
- reference numbers
- dates
- store numbers

2. Identify the actual merchant.

3. Determine the expense category.

Available categories:

{db.build_prompt()}


Important rules:

- Classify the merchant, NOT the payment provider.

Example:
PAYPAL *SPOTIFY
=> 7. abos

NOT PayPal.


- The same merchant should always receive the same category.

- Use fixed costs only for recurring necessary expenses.

- Use abos only for subscriptions.

- If uncertain, choose the closest category and lower confidence.


Examples:

REWE SAGT DANKE
=> groceries

ALDI SUED FIL.017 KIEL
=> groceries

PAYPAL *SPOTIFY
=> abos

NETFLIX.COM
=> abos

DEUTSCHE BAHN AG
=> mobility

SHELL STATION
=> mobility

APOTHEKE AM MARKT
=> healthcare


Transaction:

{description}


Return ONLY valid JSON.

Format:

{{
    "merchant": "identified merchant",
    "category": "category name",
    "confidence": 0.0
}}
"""


    response = ollama.chat(
        model="qwen2.5:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        format="json"
    )


    result = json.loads(
        response["message"]["content"]
    )


    return {

        "category": result["category"],

        "confidence": float(
            result["confidence"]
        ),

        "source": "llm"

    }



# ----------------------------------------------------------
# Merchant extraction
# ----------------------------------------------------------

def extract_merchant(description):
    """
    Simple merchant extraction.
    """

    text = description.upper()


    text = re.sub(
        r"\d+",
        "",
        text
    )


    text = re.sub(
        r"[^A-ZÄÖÜ ]",
        " ",
        text
    )


    words = text.split()


    return " ".join(
        words[:2]
    )



# ----------------------------------------------------------
# Manual correction
# ----------------------------------------------------------

def correct_transaction(df,
                        index,
                        new_category,
                        db):
    """
    Corrects a transaction manually.

    The correction is stored in the database.
    Future transactions with the same merchant
    will use this category automatically.
    """

    description = df.loc[
        index,
        "classifier"
    ]


    merchant = clean_payee(
        description
    )


    db.correct_merchant(
        merchant,
        new_category
    )

In [4]:
# ----------------------------
# Main classifier
# ----------------------------

def classify_expenses(df):

    db = ExpenseDatabase()

    results = []


    for _, row in df.iterrows():

        description = row["classifier"]


        # Classification pipeline:
        # 1. Previously corrected merchant
        # 2. Database keyword rules
        # 3. LLM fallback
        result = (
            db.lookup_memory(description)
            or
            db.apply_rules(description)
            or
            classify_with_llm(
                clean_payee(description),
                db
            )
        )


        merchant = clean_payee(description)


        # Store classification for future use
        db.save_memory(
            merchant,
            result["category"],
            result["confidence"]
        )


        results.append(result)



    output = df.copy()


    output["category"] = [
        r["category"]
        for r in results
    ]


    output["confidence"] = [
        r["confidence"]
        for r in results
    ]


    output["classification_source"] = [
        r["source"]
        for r in results
    ]


    db.close()


    return output

## Test der Pipeline mit kategorisierten Daten
Die Daten wurden synthetisch erstellt und kategorisiert.

In [5]:
transactions = DKB.load("../../data/dkb_synth_large.csv")

transactions["classifier"] = (
    transactions["payee"]
    + " "
    + transactions["description"]
)

transactions["category"] = transactions["category"].str.replace(
    r"^\d+\.\s*",
    "",
    regex=True
)

result = classify_expenses(transactions.head(30))

print(
    result[
        [
            "classifier",
            "category",
            "confidence",
            "classification_source"
        ]
    ].head(20)
)

                                           classifier       category  \
0            1und1 DSL Rechnung 06/2026 Kd-Nr. 059784    fixed costs   
1   PayPal Europe S.a.r.l. et Cie S.C.A 9946418398...       mobility   
2   Deutsche Glasfaser Rechnung 10/2025 Kd-Nr. 951...       mobility   
3   Spargelhof.Klein.0047/Düsseldorf SPARGELHOF SA...      groceries   
4   Rossmann Babywelt Lastschrift 10/2025 Ref 1705...      groceries   
5         wilhelm.tel Rechnung 06/2026 Kd-Nr. 7801104  subscriptions   
6           KVB Köln Lastschrift 09/2025 Ref 79736383    fixed costs   
7   PayPal Europe S.a.r.l. et Cie S.C.A 6355373096...       mobility   
8   PayPal Europe S.a.r.l. et Cie S.C.A 1584668793...     healthcare   
9   Hotel.Astoria.124/Köln HOTEL 2025-08-14T16:29:...  lifestyle&fun   
10  PayPal Europe S.a.r.l. et Cie S.C.A 4510173973...      groceries   
11  PayPal Europe S.a.r.l. et Cie S.C.A 6198273481...  subscriptions   
12  Book.Depository.8116/Stuttgart Kartenzahlung 2...  subscript

## Vergleich der Vorhergesagten Kategorien mit den Tatsächlichen

In [6]:
comparison = result.merge(
    transactions[["classifier", "category"]],
    on="classifier",
    suffixes=("_predicted", "_true")
)

comparison["correct"] = (
    comparison["category_predicted"] ==
    comparison["category_true"]
)

accuracy = comparison["correct"].mean()

print(f"Accuracy: {accuracy:.2%}")

correct = comparison["correct"].sum()
total = len(comparison)

print(f"Correct: {correct}/{total}")
print(f"Wrong: {total-correct}/{total}")


Accuracy: 56.67%
Correct: 17/30
Wrong: 13/30


In [13]:
correct_transaction(
    transactions,
    0,
    "Abrechnung"
)

correct_transaction(
    transactions,
    15,
    "Kleidung"
)